# IICS Activity Log Monitor

This notebook retrieves IICS / IDMC job executions, taskflow runs, and taskflow child-task details.

**Default execution window**
- Current day
- Start: **00:30 IST**
- End: **12:30 IST**
- Jobs are selected based on their **start time**

Outputs include:
- Detailed job runs
- Taskflow runs
- Taskflow child-task details
- Taskflow child-status summary
- Overall status summary
- Per-task execution summary


## 1. Install dependencies

Run this once if the packages are not already available.


In [ ]:
# Uncomment if required
# %pip install requests pandas tzdata


## 2. Configuration

For security, prefer environment variables for credentials.

Windows:
```text
set IICS_USERNAME=myuser@company.com
set IICS_PASSWORD=mypassword
```

Linux/macOS:
```text
export IICS_USERNAME="myuser@company.com"
export IICS_PASSWORD="mypassword"
```


In [ ]:
import os

# Update the login URL for your IICS POD / region if required.
LOGIN_URL = os.getenv(
    "IICS_LOGIN_URL",
    "https://dm-us.informaticacloud.com/ma/api/v2/user/login"
)

USERNAME = os.getenv("IICS_USERNAME", "")
PASSWORD = os.getenv("IICS_PASSWORD", "")

# Default monitoring window
TIMEZONE = "Asia/Kolkata"
RUN_DATE = None              # None = today in TIMEZONE
WINDOW_START = "00:30"
WINDOW_END = "12:30"

PAGE_SIZE = 1000
MAX_ROWS = None              # None = retrieve all available completed activity-log rows
LONG_RUNNING_MINUTES = 30
LATEST_COUNT = 15

OUTPUT_DIR = "iics_monitor_output"

# Set True only when you do NOT need taskflow child details.
SKIP_TASKFLOW_CHILDREN = False

print("Login URL :", LOGIN_URL)
print("Timezone  :", TIMEZONE)
print("Window    :", WINDOW_START, "to", WINDOW_END)


## 3. Imports and helper functions

In [ ]:
import os
import sys
import argparse
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import urlparse

import requests
import pandas as pd


# ---------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------

DEFAULT_LOGIN_URL = "https://dm-us.informaticacloud.com/ma/api/v2/user/login"
DEFAULT_PAGE_SIZE = 1000
REQUEST_TIMEOUT = 60

SUCCESS_STATUSES = {
    "SUCCESS", "SUCCEEDED", "COMPLETED", "COMPLETED SUCCESSFULLY"
}

FAILED_STATUSES = {
    "FAILED", "FAILURE", "ERROR", "ABORTED", "CANCELLED", "CANCELED",
    "TERMINATED"
}

WARNING_STATUSES = {
    "WARNING", "COMPLETED WITH WARNING", "COMPLETED_WITH_WARNING"
}

RUNNING_STATUSES = {
    "RUNNING", "QUEUED", "STARTING", "INITIALIZING", "SCHEDULED",
    "SUSPENDED", "WAITING"
}

TASKFLOW_TYPES = {
    "WORKFLOW", "TASKFLOW", "LINEAR TASKFLOW", "LINEAR_TASKFLOW"
}


# ---------------------------------------------------------------------
# Utility functions
# ---------------------------------------------------------------------

def safe_json(response):
    try:
        return response.json()
    except Exception:
        raise RuntimeError(
            f"Expected JSON but received HTTP {response.status_code}: "
            f"{response.text[:1000]}"
        )


def request_or_raise(method, url, **kwargs):
    try:
        response = requests.request(
            method,
            url,
            timeout=REQUEST_TIMEOUT,
            **kwargs
        )
    except requests.RequestException as exc:
        raise RuntimeError(f"HTTP request failed for {url}: {exc}") from exc

    if not response.ok:
        raise RuntimeError(
            f"HTTP {response.status_code} calling {url}\n"
            f"{response.text[:2000]}"
        )

    return response


def normalize_url(url):
    return url.rstrip("/")


def extract_list(payload):
    """
    IICS endpoints may return:
      - a list directly
      - a dictionary containing a list under a wrapper key

    This function finds the most likely list safely.
    """
    if payload is None:
        return []

    if isinstance(payload, list):
        return payload

    if isinstance(payload, dict):
        preferred_keys = [
            "activityLog",
            "activityLogEntry",
            "activityLogEntries",
            "activityMonitor",
            "activityMonitorEntry",
            "activityMonitorEntries",
            "entries",
            "items",
            "jobs",
            "results"
        ]

        for key in preferred_keys:
            value = payload.get(key)
            if isinstance(value, list):
                return value

        # Fallback: first list-valued field.
        for value in payload.values():
            if isinstance(value, list):
                return value

        # A single job object.
        if any(k in payload for k in ("runId", "taskId", "id", "status")):
            return [payload]

    return []


def first_value(row, *names, default=None):
    for name in names:
        if name in row and row[name] not in (None, ""):
            return row[name]
    return default


def parse_datetime_series(series):
    if series is None:
        return pd.Series(dtype="datetime64[ns, UTC]")
    return pd.to_datetime(series, errors="coerce", utc=True)


def classify_status(status):
    value = str(status or "").strip().upper()

    if value in SUCCESS_STATUSES:
        return "SUCCESS"
    if value in FAILED_STATUSES:
        return "FAILED"
    if value in WARNING_STATUSES:
        return "WARNING"
    if value in RUNNING_STATUSES:
        return "RUNNING"
    if not value:
        return "UNKNOWN"
    return "OTHER"


def classify_task_type(task_type):
    value = str(task_type or "").strip().upper()
    if value in TASKFLOW_TYPES or "TASKFLOW" in value or "WORKFLOW" in value:
        return "TASKFLOW"
    return "TASK"


def seconds_to_text(seconds):
    if pd.isna(seconds):
        return ""
    seconds = int(max(0, seconds))
    hours, rem = divmod(seconds, 3600)
    minutes, secs = divmod(rem, 60)
    if hours:
        return f"{hours:02d}:{minutes:02d}:{secs:02d}"
    return f"{minutes:02d}:{secs:02d}"


# ---------------------------------------------------------------------
# IICS REST client
# ---------------------------------------------------------------------

class IICSClient:
    def __init__(self, login_url, username, password):
        self.login_url = login_url
        self.username = username
        self.password = password
        self.session_id = None
        self.server_url = None

    def login(self):
        payload = {
            "@type": "login",
            "username": self.username,
            "password": self.password
        }

        response = request_or_raise(
            "POST",
            self.login_url,
            headers={
                "Content-Type": "application/json",
                "Accept": "application/json"
            },
            json=payload
        )

        data = safe_json(response)

        self.session_id = data.get("icSessionId")
        self.server_url = data.get("serverUrl")

        if not self.session_id or not self.server_url:
            raise RuntimeError(
                "Login succeeded but icSessionId/serverUrl was not returned. "
                "Check the IICS login endpoint and account authentication method."
            )

        self.server_url = normalize_url(self.server_url)
        return data

    @property
    def headers(self):
        if not self.session_id:
            raise RuntimeError("Not logged in.")

        return {
            "Accept": "application/json",
            "Content-Type": "application/json",
            "icSessionId": self.session_id
        }

    def get_completed_jobs(self, page_size=DEFAULT_PAGE_SIZE, max_rows=None):
        """
        Retrieve completed job executions through activityLog.
        The API supports rowLimit up to 1000 and offset pagination.
        """
        page_size = min(max(int(page_size), 1), 1000)

        url = f"{self.server_url}/api/v2/activity/activityLog"
        all_rows = []
        offset = 0

        while True:
            remaining = None if max_rows is None else max_rows - len(all_rows)
            if remaining is not None and remaining <= 0:
                break

            current_limit = page_size
            if remaining is not None:
                current_limit = min(current_limit, remaining)

            response = request_or_raise(
                "GET",
                url,
                headers=self.headers,
                params={
                    "offset": offset,
                    "rowLimit": current_limit
                }
            )

            payload = safe_json(response)
            rows = extract_list(payload)

            if not rows:
                break

            all_rows.extend(rows)

            if len(rows) < current_limit:
                break

            offset += len(rows)

        return all_rows

    def get_running_jobs(self, details=True):
        """
        Retrieve currently running jobs through activityMonitor.
        """
        url = f"{self.server_url}/api/v2/activity/activityMonitor"

        response = request_or_raise(
            "GET",
            url,
            headers=self.headers,
            params={"details": str(bool(details)).lower()}
        )

        return extract_list(safe_json(response))


# ---------------------------------------------------------------------
# Dataset normalization
# ---------------------------------------------------------------------

def normalize_jobs(completed_rows, running_rows):
    records = []

    def append_rows(rows, source):
        for row in rows:
            if not isinstance(row, dict):
                continue

            task_type = first_value(
                row,
                "type", "taskType", "assetType", "jobType",
                default=""
            )

            status = first_value(
                row,
                "status", "runStatus", "state",
                default="RUNNING" if source == "activityMonitor" else ""
            )

            record = {
                "source": source,

                "log_id": first_value(
                    row, "id", "activityLogId", "logId"
                ),

                "run_id": first_value(
                    row, "runId", "jobId", "executionId"
                ),

                "task_id": first_value(
                    row, "taskId", "assetId", "objectId"
                ),

                "task_name": first_value(
                    row,
                    "taskName", "name", "assetName", "objectName",
                    default=""
                ),

                "task_type": task_type,
                "task_category": classify_task_type(task_type),

                "status_raw": status,
                "status_group": classify_status(status),

                "started_by": first_value(
                    row, "startedBy", "runBy", "userName",
                    default=""
                ),

                "start_time": first_value(
                    row, "startTime", "startDate", "startedAt"
                ),

                "end_time": first_value(
                    row, "endTime", "endDate", "endedAt"
                ),

                "runtime_environment": first_value(
                    row,
                    "runtimeEnvironmentName", "runtimeEnvironment",
                    "runtimeEnvName", "runtimeEnv",
                    default=""
                ),

                "agent_name": first_value(
                    row, "agentName", "secureAgentName",
                    default=""
                ),

                "message": first_value(
                    row,
                    "messageText", "errorMessage", "message",
                    default=""
                ),

                "success_source_rows": first_value(
                    row,
                    "successSourceRows", "successfulSourceRows",
                    "srcSuccessRows"
                ),

                "failed_source_rows": first_value(
                    row,
                    "failedSourceRows", "srcFailedRows"
                ),

                "success_target_rows": first_value(
                    row,
                    "successTargetRows", "successfulTargetRows",
                    "tgtSuccessRows"
                ),

                "failed_target_rows": first_value(
                    row,
                    "failedTargetRows", "tgtFailedRows"
                ),

                "raw_record": row
            }

            records.append(record)

    append_rows(completed_rows, "activityLog")
    append_rows(running_rows, "activityMonitor")

    df = pd.DataFrame(records)

    if df.empty:
        return df

    df["start_time_utc"] = parse_datetime_series(df["start_time"])
    df["end_time_utc"] = parse_datetime_series(df["end_time"])

    now_utc = pd.Timestamp.now(tz="UTC")

    effective_end = df["end_time_utc"].copy()
    running_mask = effective_end.isna()
    effective_end.loc[running_mask] = now_utc

    df["duration_seconds"] = (
        effective_end - df["start_time_utc"]
    ).dt.total_seconds()

    df["duration"] = df["duration_seconds"].apply(seconds_to_text)

    # Avoid duplicates if a job moves from Monitor to Activity Log while the
    # script is executing.
    dedupe_cols = [c for c in ["task_id", "run_id"] if c in df.columns]

    if dedupe_cols:
        # Prefer activityLog record over activityMonitor when duplicate exists.
        df["_source_priority"] = df["source"].map({
            "activityMonitor": 1,
            "activityLog": 2
        }).fillna(0)

        df = (
            df.sort_values("_source_priority")
              .drop_duplicates(subset=dedupe_cols, keep="last")
              .drop(columns="_source_priority")
        )

    df = df.sort_values(
        "start_time_utc",
        ascending=False,
        na_position="last"
    ).reset_index(drop=True)

    return df


# ---------------------------------------------------------------------
# Summary and reporting
# ---------------------------------------------------------------------

def print_title(text):
    print("\n" + "=" * 80)
    print(text)
    print("=" * 80)


def value_count(df, column, value):
    if df.empty or column not in df:
        return 0
    return int((df[column] == value).sum())


def quick_summary(df, latest_count=15, long_running_minutes=30):
    print_title("IICS QUICK SUMMARY")

    if df.empty:
        print("No job executions were returned.")
        return

    total = len(df)
    success = value_count(df, "status_group", "SUCCESS")
    failed = value_count(df, "status_group", "FAILED")
    warning = value_count(df, "status_group", "WARNING")
    running = value_count(df, "status_group", "RUNNING")
    other = total - success - failed - warning - running

    completed_for_rate = success + failed + warning
    success_rate = (
        (success / completed_for_rate * 100.0)
        if completed_for_rate else 0.0
    )

    taskflows = df[df["task_category"] == "TASKFLOW"].copy()

    print(f"Total job runs       : {total:,}")
    print(f"Successful           : {success:,}")
    print(f"Failed               : {failed:,}")
    print(f"Warnings             : {warning:,}")
    print(f"Running / active     : {running:,}")
    print(f"Other / unknown      : {other:,}")
    print(f"Success rate         : {success_rate:,.2f}%")
    print(f"Taskflow runs        : {len(taskflows):,}")

    if not taskflows.empty:
        tf_status = (
            taskflows["status_group"]
            .value_counts(dropna=False)
            .rename_axis("status")
            .reset_index(name="runs")
        )

        print("\nTaskflow status:")
        print(tf_status.to_string(index=False))

    failed_df = df[df["status_group"] == "FAILED"].copy()

    if not failed_df.empty:
        print_title(f"FAILED JOBS ({len(failed_df):,})")
        cols = [
            "task_name", "task_type", "run_id",
            "start_time_utc", "duration", "message"
        ]
        cols = [c for c in cols if c in failed_df.columns]
        print(
            failed_df[cols]
            .head(25)
            .to_string(index=False, max_colwidth=100)
        )

    long_running = df[
        (df["status_group"] == "RUNNING") &
        (df["duration_seconds"] >= long_running_minutes * 60)
    ].copy()

    if not long_running.empty:
        print_title(
            f"LONG-RUNNING ACTIVE JOBS "
            f"(>= {long_running_minutes} MINUTES)"
        )
        cols = [
            "task_name", "task_type", "run_id",
            "start_time_utc", "duration", "runtime_environment"
        ]
        cols = [c for c in cols if c in long_running.columns]
        print(long_running[cols].to_string(index=False))

    print_title(f"LATEST {min(latest_count, len(df))} EXECUTIONS")

    cols = [
        "task_name",
        "task_type",
        "status_group",
        "run_id",
        "start_time_utc",
        "duration",
        "started_by"
    ]
    cols = [c for c in cols if c in df.columns]

    print(
        df[cols]
        .head(latest_count)
        .to_string(index=False)
    )


def create_summary_tables(df):
    if df.empty:
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

    by_status = (
        df.groupby(["task_category", "status_group"], dropna=False)
          .size()
          .reset_index(name="run_count")
          .sort_values(["task_category", "run_count"], ascending=[True, False])
    )

    by_task = (
        df.groupby(["task_name", "task_type"], dropna=False)
          .agg(
              total_runs=("run_id", "size"),
              successes=("status_group",
                         lambda s: int((s == "SUCCESS").sum())),
              failures=("status_group",
                        lambda s: int((s == "FAILED").sum())),
              warnings=("status_group",
                        lambda s: int((s == "WARNING").sum())),
              running=("status_group",
                       lambda s: int((s == "RUNNING").sum())),
              avg_duration_seconds=("duration_seconds", "mean"),
              max_duration_seconds=("duration_seconds", "max"),
              latest_start=("start_time_utc", "max")
          )
          .reset_index()
    )

    denominator = by_task["successes"] + by_task["failures"] + by_task["warnings"]

    by_task["success_rate_pct"] = (
        by_task["successes"]
        .div(denominator.where(denominator != 0))
        .mul(100)
        .fillna(0)
        .round(2)
    )

    by_task["avg_duration"] = (
        by_task["avg_duration_seconds"].apply(seconds_to_text)
    )
    by_task["max_duration"] = (
        by_task["max_duration_seconds"].apply(seconds_to_text)
    )

    by_task = by_task.sort_values(
        ["failures", "total_runs"],
        ascending=False
    )

    taskflow_runs = (
        df[df["task_category"] == "TASKFLOW"]
        .copy()
        .sort_values("start_time_utc", ascending=False)
    )

    return by_status, by_task, taskflow_runs


def write_outputs(df, output_dir):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    detailed_path = output_dir / f"iics_job_runs_{timestamp}.csv"
    taskflows_path = output_dir / f"iics_taskflow_runs_{timestamp}.csv"
    status_path = output_dir / f"iics_status_summary_{timestamp}.csv"
    task_summary_path = output_dir / f"iics_task_summary_{timestamp}.csv"

    export_df = df.copy()

    # raw_record is useful in memory but ugly in CSV.
    if "raw_record" in export_df.columns:
        export_df["raw_record"] = export_df["raw_record"].astype(str)

    export_df.to_csv(detailed_path, index=False)

    by_status, by_task, taskflow_runs = create_summary_tables(df)

    by_status.to_csv(status_path, index=False)
    by_task.to_csv(task_summary_path, index=False)
    taskflow_runs.to_csv(taskflows_path, index=False)

    return {
        "job_runs": detailed_path,
        "taskflow_runs": taskflows_path,
        "status_summary": status_path,
        "task_summary": task_summary_path
    }

## 4. Resolve the default IST execution window

The notebook uses **today 00:30 to 12:30 IST** unless you change `RUN_DATE`, `WINDOW_START`, or `WINDOW_END`.


In [ ]:
from datetime import datetime
from zoneinfo import ZoneInfo

def resolve_window(run_date=None, start_hhmm="00:30", end_hhmm="12:30", timezone_name="Asia/Kolkata"):
    tz = ZoneInfo(timezone_name)
    now = datetime.now(tz)

    if run_date is None:
        date_part = now.date()
    else:
        date_part = pd.to_datetime(run_date).date()

    start_t = datetime.strptime(start_hhmm, "%H:%M").time()
    end_t = datetime.strptime(end_hhmm, "%H:%M").time()

    start_local = datetime.combine(date_part, start_t, tzinfo=tz)
    end_local = datetime.combine(date_part, end_t, tzinfo=tz)

    if end_local <= start_local:
        end_local = end_local + pd.Timedelta(days=1)

    return pd.Timestamp(start_local), pd.Timestamp(end_local)

WINDOW_START_TS, WINDOW_END_TS = resolve_window(
    RUN_DATE,
    WINDOW_START,
    WINDOW_END,
    TIMEZONE
)

print("Execution window:")
print(" Start:", WINDOW_START_TS)
print(" End  :", WINDOW_END_TS)


## 5. Login and pull completed + active jobs


In [ ]:
if not USERNAME or not PASSWORD:
    raise ValueError(
        "IICS_USERNAME / IICS_PASSWORD are not set. "
        "Set environment variables or populate USERNAME and PASSWORD in the configuration cell."
    )

client = IICSClient(
    login_url=LOGIN_URL,
    username=USERNAME,
    password=PASSWORD
)

login_info = client.login()

print("Connected to:", client.server_url)

completed_rows = client.get_completed_jobs(
    page_size=PAGE_SIZE,
    max_rows=MAX_ROWS
)

running_rows = client.get_running_jobs(details=True)

print("Completed rows retrieved:", len(completed_rows))
print("Running rows retrieved  :", len(running_rows))


## 6. Normalize and filter to 00:30–12:30 IST

Filtering is applied using the job start timestamp.


In [ ]:
jobs_df = normalize_jobs(completed_rows, running_rows)

if jobs_df.empty:
    filtered_jobs_df = jobs_df.copy()
else:
    start_ist = jobs_df["start_time_utc"].dt.tz_convert(TIMEZONE)

    filtered_jobs_df = jobs_df[
        (start_ist >= WINDOW_START_TS) &
        (start_ist < WINDOW_END_TS)
    ].copy()

    filtered_jobs_df["start_time_ist"] = (
        filtered_jobs_df["start_time_utc"].dt.tz_convert(TIMEZONE)
    )

    filtered_jobs_df["end_time_ist"] = (
        filtered_jobs_df["end_time_utc"].dt.tz_convert(TIMEZONE)
    )

print("Jobs in selected window:", len(filtered_jobs_df))
display(filtered_jobs_df.head(20))


## 7. Taskflow child-task details

This section uses detailed activity-log records for completed taskflows and detailed activity-monitor records for currently active taskflows.

If your current script version contains dedicated child-expansion helper functions, they are used here automatically. Otherwise the notebook falls back to recursively extracting `entries` / child arrays from the available raw records.


In [ ]:
def _child_arrays(obj):
    if not isinstance(obj, dict):
        return []

    candidate_keys = [
        "entries",
        "children",
        "childTasks",
        "subTasks",
        "subtasks",
        "tasks",
        "steps"
    ]

    arrays = []
    for key in candidate_keys:
        value = obj.get(key)
        if isinstance(value, list):
            arrays.extend(value)
    return arrays


def extract_children_from_record(parent_row, taskflow_name="", taskflow_run_id=None,
                                 taskflow_status="", level=1, parent_child_name=""):
    rows = []

    for child in _child_arrays(parent_row):
        if not isinstance(child, dict):
            continue

        child_name = first_value(
            child, "taskName", "name", "assetName", "objectName", "stepName",
            default=""
        )
        child_type = first_value(
            child, "type", "taskType", "assetType", "jobType",
            default=""
        )
        child_status_raw = first_value(
            child, "status", "runStatus", "state",
            default=""
        )

        start_time = first_value(
            child, "startTime", "startDate", "startedAt"
        )
        end_time = first_value(
            child, "endTime", "endDate", "endedAt"
        )

        start_ts = pd.to_datetime(start_time, errors="coerce", utc=True)
        end_ts = pd.to_datetime(end_time, errors="coerce", utc=True)

        if pd.isna(end_ts) and not pd.isna(start_ts):
            effective_end = pd.Timestamp.now(tz="UTC")
        else:
            effective_end = end_ts

        duration_seconds = None
        if not pd.isna(start_ts) and not pd.isna(effective_end):
            duration_seconds = (effective_end - start_ts).total_seconds()

        row = {
            "taskflow_name": taskflow_name,
            "taskflow_run_id": taskflow_run_id,
            "taskflow_status": taskflow_status,
            "level": level,
            "parent_child_name": parent_child_name,
            "child_name": child_name,
            "child_type": child_type,
            "child_run_id": first_value(
                child, "runId", "jobId", "executionId"
            ),
            "child_task_id": first_value(
                child, "taskId", "assetId", "objectId"
            ),
            "child_status_raw": child_status_raw,
            "child_status": classify_status(child_status_raw),
            "start_time_utc": start_ts,
            "end_time_utc": end_ts,
            "duration_seconds": duration_seconds,
            "duration": seconds_to_text(duration_seconds),
            "runtime_environment": first_value(
                child,
                "runtimeEnvironmentName", "runtimeEnvironment",
                "runtimeEnvName", "runtimeEnv",
                default=""
            ),
            "success_source_rows": first_value(
                child, "successSourceRows", "successfulSourceRows", "srcSuccessRows"
            ),
            "failed_source_rows": first_value(
                child, "failedSourceRows", "srcFailedRows"
            ),
            "success_target_rows": first_value(
                child, "successTargetRows", "successfulTargetRows", "tgtSuccessRows"
            ),
            "failed_target_rows": first_value(
                child, "failedTargetRows", "tgtFailedRows"
            ),
            "message": first_value(
                child, "messageText", "errorMessage", "message",
                default=""
            ),
            "raw_record": child
        }

        rows.append(row)

        rows.extend(
            extract_children_from_record(
                child,
                taskflow_name=taskflow_name,
                taskflow_run_id=taskflow_run_id,
                taskflow_status=taskflow_status,
                level=level + 1,
                parent_child_name=child_name
            )
        )

    return rows


def build_taskflow_children(filtered_df):
    all_children = []

    taskflows = filtered_df[
        filtered_df["task_category"] == "TASKFLOW"
    ].copy()

    if taskflows.empty:
        return pd.DataFrame()

    for _, tf in taskflows.iterrows():
        raw = tf.get("raw_record")
        if not isinstance(raw, dict):
            continue

        children = extract_children_from_record(
            raw,
            taskflow_name=tf.get("task_name", ""),
            taskflow_run_id=tf.get("run_id"),
            taskflow_status=tf.get("status_group", "")
        )
        all_children.extend(children)

    child_df = pd.DataFrame(all_children)

    if not child_df.empty:
        child_df["start_time_ist"] = pd.to_datetime(
            child_df["start_time_utc"], errors="coerce", utc=True
        ).dt.tz_convert(TIMEZONE)

        child_df["end_time_ist"] = pd.to_datetime(
            child_df["end_time_utc"], errors="coerce", utc=True
        ).dt.tz_convert(TIMEZONE)

    return child_df


if SKIP_TASKFLOW_CHILDREN:
    taskflow_children_df = pd.DataFrame()
else:
    # First use any dedicated implementation included in the source script, if present.
    dedicated_fn = None
    for fn_name in [
        "get_taskflow_child_details",
        "extract_taskflow_children",
        "build_taskflow_child_details",
        "get_all_taskflow_children"
    ]:
        if fn_name in globals() and callable(globals()[fn_name]):
            dedicated_fn = globals()[fn_name]
            break

    if dedicated_fn:
        try:
            taskflow_children_df = dedicated_fn(client, filtered_jobs_df)
        except TypeError:
            try:
                taskflow_children_df = dedicated_fn(filtered_jobs_df)
            except Exception:
                taskflow_children_df = build_taskflow_children(filtered_jobs_df)
    else:
        taskflow_children_df = build_taskflow_children(filtered_jobs_df)

print("Taskflow child records:", len(taskflow_children_df))
display(taskflow_children_df.head(30))


## 8. Quick operational summary

In [ ]:
quick_summary(
    filtered_jobs_df,
    latest_count=LATEST_COUNT,
    long_running_minutes=LONG_RUNNING_MINUTES
)

if not taskflow_children_df.empty:
    print("\nTASKFLOW CHILD STATUS SUMMARY")
    print("=" * 80)

    child_status_summary = (
        taskflow_children_df["child_status"]
        .value_counts(dropna=False)
        .rename_axis("status")
        .reset_index(name="runs")
    )

    display(child_status_summary)

    failed_children_df = taskflow_children_df[
        taskflow_children_df["child_status"] == "FAILED"
    ].copy()

    if not failed_children_df.empty:
        print("\nFAILED TASKFLOW CHILD TASKS")
        display(
            failed_children_df[
                [
                    c for c in [
                        "taskflow_name",
                        "taskflow_run_id",
                        "child_name",
                        "child_type",
                        "child_status",
                        "start_time_ist",
                        "duration",
                        "message"
                    ]
                    if c in failed_children_df.columns
                ]
            ]
        )


## 9. Summary tables

In [ ]:
status_summary_df, task_summary_df, taskflow_runs_df = create_summary_tables(
    filtered_jobs_df
)

display(status_summary_df)
display(task_summary_df.head(50))
display(taskflow_runs_df.head(50))


## 10. Export CSV outputs

In [ ]:
from pathlib import Path

output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

paths = {}

paths["job_runs"] = output_dir / f"iics_job_runs_{timestamp}.csv"
filtered_jobs_df.to_csv(paths["job_runs"], index=False)

paths["taskflow_runs"] = output_dir / f"iics_taskflow_runs_{timestamp}.csv"
taskflow_runs_df.to_csv(paths["taskflow_runs"], index=False)

paths["status_summary"] = output_dir / f"iics_status_summary_{timestamp}.csv"
status_summary_df.to_csv(paths["status_summary"], index=False)

paths["task_summary"] = output_dir / f"iics_task_summary_{timestamp}.csv"
task_summary_df.to_csv(paths["task_summary"], index=False)

paths["taskflow_child_tasks"] = output_dir / f"iics_taskflow_child_tasks_{timestamp}.csv"
taskflow_children_df.to_csv(paths["taskflow_child_tasks"], index=False)

if not taskflow_children_df.empty:
    child_status_summary = (
        taskflow_children_df
        .groupby(
            ["taskflow_name", "child_name", "child_type", "child_status"],
            dropna=False
        )
        .size()
        .reset_index(name="run_count")
        .sort_values(["taskflow_name", "child_name", "run_count"],
                     ascending=[True, True, False])
    )
else:
    child_status_summary = pd.DataFrame()

paths["taskflow_child_status_summary"] = (
    output_dir / f"iics_taskflow_child_status_summary_{timestamp}.csv"
)
child_status_summary.to_csv(
    paths["taskflow_child_status_summary"],
    index=False
)

print("Generated files:")
for name, path in paths.items():
    print(f"{name:35s}: {path}")


## 11. Optional: view only failures

Use these cells for a quick daily support view.


In [ ]:
failed_jobs_df = filtered_jobs_df[
    filtered_jobs_df["status_group"] == "FAILED"
].copy()

display(
    failed_jobs_df[
        [
            c for c in [
                "task_name",
                "task_type",
                "run_id",
                "start_time_ist",
                "duration",
                "message"
            ]
            if c in failed_jobs_df.columns
        ]
    ]
)
